# QuantumTrain 工具書 02：設計可信的電路深度實驗

先固定問題與預算，再執行、檢查完整性、閱讀曲線並匯出。

## 0. 研究問題與閱讀地圖

**本節目標**：明確定義要回答的問題。

**為什麼需要這一步**：在跑數據前先決定變因，避免事後挑故事。

**輸入／輸出與 shape**：設定 → 電路深度比較；target 固定 64→8→10。

**關鍵原理與程式定位**：同 epoch 預算下，1/3/5 層的準確率與成本如何變化？延伸自 workbook 01，使用相同模組。

**操作步驟**：先完成第一份教材，再執行 setup；預設 smoke。

**預期觀察**：印出模式；full 才會跑 4 組×3 seeds×20 epochs。

**結果判讀與常見錯誤**：這是小任務適配，深度與參數量同時變動，不能宣稱單獨的深度因果效應。

**理解問題**：研究的成功是否要求更深一定更好？

<details><summary>提示與解答</summary>

不需要；穩定地觀察沒有改善或成本增加也有價值。

</details>


In [ ]:
import os, sys, json, time
from pathlib import Path
import torch, pennylane as qml
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
from torch.utils.data import DataLoader
from qt_eval.config import load_config, validate_config
from qt_eval.data import load_digits_splits
from qt_eval.models import make_model, ClassicalMLP, functional_mlp
from qt_eval.quantum import QuantumTrain, make_qnode, basis_codes, probability_features
from qt_eval.diagnostics import check_training_step
from qt_eval.experiment import run_experiment, evaluate
from qt_eval.reporting import summarize_runs, write_report
from qt_eval.artifacts import SOURCE_COMMIT

# 教學預設 smoke；只有準備好正式預算後才手動改成 full。
MODE = os.environ.get('QT_EVAL_MODE', 'smoke')
ROOT = Path.cwd()
if not (ROOT / 'pyproject.toml').exists():
    ROOT = ROOT.parent
assert (ROOT / 'pyproject.toml').exists(), '請由專案根目錄或 notebooks 目錄啟動'
OUTPUT = Path(os.environ.get('QT_EVAL_OUTPUT', str(ROOT/'results'/'notebooks')))
CONFIG = load_config(ROOT/'configs'/('smoke.json' if MODE=='smoke' else 'depth_study.json'))
assert CONFIG['mode'] == MODE
torch.set_num_threads(CONFIG['torch_threads'])
torch.manual_seed(CONFIG['seed'])
print('MODE:', MODE, '| device: cpu | dtype: float64')
if MODE == 'smoke':
    print('工程檢查：小資料、短訓練，不可用來判斷模型研究表現。')

## 1. 凍結比較矩陣

**本節目標**：理解固定項與變動項。

**為什麼需要這一步**：如果資料或 optimizer 一起變，難以解釋差異。

**輸入／輸出與 shape**：4 組 × seeds → 4（smoke）或12（full）份配置。

**關鍵原理與程式定位**：固定 split_seed、LR、batch、epochs、mapping；seed 只控制初始化與 shuffle。

**操作步驟**：列印全部矩陣；正式執行前檢查設定。

**預期觀察**：classical、quantum-depth1/3/5，full seeds=11/22/33。

**結果判讀與常見錯誤**：相同 seed 不代表不同深度模型具有完全相同初始化；這是重跑識別，不是等價模型配對。

**理解問題**：為什麼保持 split_seed 不變？

<details><summary>提示與解答</summary>

避免把不同測試資料難度混入初始化變異。

</details>


In [ ]:
from qt_eval.study import study_configs, GROUPS
matrix=study_configs(CONFIG)
display(pd.DataFrame(matrix)[['mode','model','q_depth','epochs','seed','split_seed']])

## 2. 用 pilot 估計成本

**本節目標**：在完整矩陣前知道預算。

**為什麼需要這一步**：深電路的每步成本可能更高；先測再決定正式運算。

**輸入／輸出與 shape**：最深設定的短 pilot → 每 epoch 時間與矩陣粗估。

**關鍵原理與程式定位**：pilot 用 full 資料但只1 epoch（smoke 則小資料）；固定 seed，結果不併入正式報告。

**操作步驟**：執行1 epoch pilot；full 保持 validation-only，不讀 test；短測後不調參。

**預期觀察**：顯示 train-only 粗估，還有 validation、I/O 等額外成本。

**結果判讀與常見錯誤**：最深模型的估算不是精確總耗時；pilot 的 test 不用於設定決策。

**理解問題**：能用 pilot test 選深度嗎？

<details><summary>提示與解答</summary>

不能；本 pilot 只執行 fit(train,validation)，根本不接 test。

</details>


In [ ]:
from qt_eval.study import pilot_training
pilot=pilot_training(CONFIG,OUTPUT/'pilot')
print(pilot)
print('Rough upper-style train-only estimate:',pilot['train_seconds']*CONFIG['epochs']*len(matrix),'seconds')

## 3. 批次執行與 artifact

**本節目標**：把矩陣變成可追蹤 runs。

**為什麼需要這一步**：逐 run 保存，才能保留失敗原因而不只剩最好的數字。

**輸入／輸出與 shape**：配置列表 → manifest 與每次 run 目錄。

**關鍵原理與程式定位**：study.run_study 順序執行；報告只接此矩陣的 run 清單，不掃入舊 runs。

**操作步驟**：執行當前模式；需真正完整實驗時在 setup 明確設 full 再 Restart & Run All。

**預期觀察**：每 run 顯示完成或失敗，保存 study.json 與獨立結果。

**結果判讀與常見錯誤**：重新執行會建立新 study，不覆蓋歷史。不可替換表現差的 seed。

**理解問題**：若有一個 run 失敗，該怎麼報告？

<details><summary>提示與解答</summary>

保留 failed 與原因，組別標 incomplete；先診斷而非只刪除該筆。

</details>


In [ ]:
from qt_eval.study import run_study
study_dir=run_study(CONFIG,OUTPUT/'studies')
manifest=json.loads((study_dir/'study.json').read_text())
runs=[study_dir/p for p in manifest['runs']]
print('Study:',study_dir,'| attempted:',len(runs))

## 4. 完整性與缺漏

**本節目標**：辨認缺 seed、整組缺失與重複執行。

**為什麼需要這一步**：只平均成功的幾筆容易形成選擇偏差。

**輸入／輸出與 shape**：run records → expected/completed/failed/missing_seeds。

**關鍵原理與程式定位**：reporting.summarize_runs 拒絕重複 group/seed 和不合協定；空組也顯示。

**操作步驟**：查看摘要是否每組完整；smoke 必須顯式 opt-in。

**預期觀察**：smoke 每組 expected=1；full expected=3。

**結果判讀與常見錯誤**：completed 不代表模型效果好，只代表流程與保存成功。

**理解問題**：一個 seed 的標準差能填0嗎？

<details><summary>提示與解答</summary>

不能；沒有跨執行變動的估計，應顯示 N/A。

</details>


In [ ]:
summary=summarize_runs(runs,allow_smoke=MODE=='smoke',expected_groups=GROUPS)
display(summary)
print('All planned runs complete:',bool((summary.status=='complete').all()))

## 5. 精度、曲線與時間

**本節目標**：一起看任務結果與代價。

**為什麼需要這一步**：參數量少不代表訓練快，loss 好也不代表泛化好。

**輸入／輸出與 shape**：每 epoch 紀錄與 final test → 曲線、均值、樣本標準差。

**關鍵原理與程式定位**：reporting.py 的圖表保留每個 seed，accuracy 的誤差棒是 ddof=1 標準差。

**操作步驟**：先讀每組各次結果，再看自動報告；切勿用此 test 結果調整同一輪模型。

**預期觀察**：loss 軸是平均 cross entropy；時間單位為秒；單 seed 不畫誤差棒。

**結果判讀與常見錯誤**：若差距小於 seed 波動，結論應保守；三次不構成強統計證據。

**理解問題**：為什麼要留下單 seed 曲線？

<details><summary>提示與解答</summary>

平均會掩蓋不穩定或失敗的訓練過程。

</details>


In [ ]:
from IPython.display import Image
report=write_report(runs,study_dir/'report',allow_smoke=MODE=='smoke',expected_groups=GROUPS)
for image in ('train_loss.png','val_loss.png','val_accuracy.png','accuracy_mean.png','train_seconds_mean.png'):
    display(Image(filename=str(report.parent/image)))
print('Caption: epoch curves are individual runs; bar charts aggregate completed runs. Accuracy 0–1; time seconds; error bars sample SD.')

## 6. 梯度與成本的界線

**本節目標**：避免把健康檢查誤認為研究突破。

**為什麼需要這一步**：有梯度只是更新所需條件，不保證量子分支有幫助。

**輸入／輸出與 shape**：diagnostics＋parameter counts → 每組的更新證據。

**關鍵原理與程式定位**：量子、mapping 登記參數量與 target weights 分開；讀取每次 run 的診斷。

**操作步驟**：比對 quantum norm/delta 與訓練時間。

**預期觀察**：量子組有有效更新；classical 沒有量子欄位。

**結果判讀與常見錯誤**：backprop 理想模擬不等於 QPU；此 mapping 大於 target，不可稱壓縮。

**理解問題**：如果量子梯度很小，能直接叫 barren plateau 嗎？

<details><summary>提示與解答</summary>

不能。要先檢查初始化、對稱性、尺度、有效參數、連線，再設計專門的縮放實驗。

</details>


In [ ]:
for path in runs:
    diagnostic=path/'diagnostics.json'
    if diagnostic.exists():
        print(path.name,json.loads(diagnostic.read_text())['groups'])
display(summary[['group','parameters','train_seconds_mean']])

## 7. 匯出與再現

**本節目標**：由既有結果重新產生相同數字。

**為什麼需要這一步**：報告不是手打表格；讀者需要能追到 runs 與配置。

**輸入／輸出與 shape**：既有 artifacts → depth-study.md、summary.csv、manifest.json 與PNG。

**關鍵原理與程式定位**：write_report 不訓練，只重建報告；README 說明路徑。

**操作步驟**：開啟 Markdown 報告，核對 manifest 中的來源 run ids。

**預期觀察**：數據與第4節一致，圖表來自第5節。

**結果判讀與常見錯誤**：複製報告時一併保存 manifest 與圖表；本機完整 checkpoint 留在 results。

**理解問題**：為什麼不掃 results 下的所有目錄？

<details><summary>提示與解答</summary>

可能混入 pilot、不同設定或重複 seed；應用明確的矩陣 manifest。

</details>


In [ ]:
display(Markdown(report.read_text(encoding='utf-8')))
print('Report:',report)
print('Configuration manifest:',report.parent/'manifest.json')

## 8. 寫出有證據的結論

**本節目標**：把結果轉成可核對的技術敘述。

**為什麼需要這一步**：作品價值在正確解釋，不在強行找到優勢。

**輸入／輸出與 shape**：數據＋限制 → 觀察、證據、下一個實驗。

**關鍵原理與程式定位**：先描述哪組的精度和時間，再交代 seeds、模擬器與目標；不要推論未做的比較。

**操作步驟**：閱讀限制；若是 full，依自己的實測寫三句觀察與一項下一步。

**預期觀察**：smoke 只得出流程檢查結論；full 才討論本設定的表現。

**結果判讀與常見錯誤**：不能用「量子比較好」取代指定指標與條件；若接著改 mapping，需另起研究協定。

**理解問題**：最直接測量量子學習貢獻的下一步？

<details><summary>提示與解答</summary>

相同條件下固定量子參數，只訓練 mapping，做消融對照。

</details>


In [ ]:
display(Markdown((ROOT/'docs/methodology.md').read_text(encoding='utf-8')))
print('Conclusion scope:', 'pipeline checks only' if MODE=='smoke' else 'this fixed digits protocol only')